In [65]:
keywords = [
    # Macro economie
    "inflation", "deflation", "interest rates", "rate hike", "rate cut",
    "recession", "economic slowdown", "gdp", "consumer spending",
    "unemployment", "job growth", "employment", "wage growth",
    "housing market", "real estate", "credit", "debt", "liquidity",

    # Centrale banken / beleid
    "federal reserve", "fed", "ecb", "central bank",
    "monetary policy", "quantitative easing", "quantitative tightening",
    "bond yields", "treasury yields",

    # Markt / trading termen
    "stock market", "stocks", "equities", "bull market", "bear market",
    "market rally", "market crash", "selloff", "volatility", "correction",
    "overvalued", "undervalued", "bubble",

    # Bedrijven / earnings
    "earnings", "earnings report", "revenue", "profit", "guidance",
    "forecast", "downgrade", "upgrade", "ipo", "merger", "acquisition",

    # Tech / AI (super belangrijk momenteel)
    "ai", "artificial intelligence", "machine learning", "automation",
    "semiconductors", "chips", "nvidia", "openai", "cloud computing",

    # Grote namen (markt movers)
    "elon musk", "tesla", "apple", "microsoft", "amazon", "google", "meta",

    # Politiek / geopolitiek
    "trump", "biden", "white house", "election",
    "war", "conflict", "sanctions", "china", "russia", "ukraine",
    "middle east", "trade war", "tariffs",

    # Grondstoffen / alternatieven
    "oil", "gold", "commodities", "energy prices", "gas prices",

    # Sentiment / angst
    "fear", "panic", "uncertainty", "risk", "risk-off", "risk-on",
    "investor sentiment",

    # Crypto (vaak leading indicator voor risk appetite)
    "bitcoin", "crypto", "cryptocurrency", "blockchain",

    # Banken / financiële stress
    "banking crisis", "bank failure", "liquidity crisis",
    "credit crunch", "default"
]

sections = ["Business Day", "Health", "Education", "Science", "Blogs", "U.S.", "New York", "Real Estate", "Washington", "World", "Your Money", "Technology", "Job Market"]

In [66]:
import requests
import time
import os
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
import re

load_dotenv()

API_KEY = os.getenv("NYT_API_KEY")
START_YEAR = 2006
CURRENT_YEAR = datetime.now().year
CURRENT_MONTH = datetime.now().month

def fetch_and_save_nyt_data():
    for year in range(START_YEAR, 2007):
        year_data = []
        
        # Bepaal tot welke maand we moeten gaan voor het huidige jaar
        end_month = CURRENT_MONTH if year == CURRENT_YEAR else 12
        
        for month in range(1, end_month + 1):
            print(f"Ophalen: {year}-{month}...")
            
            url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json?api-key={API_KEY}"
            
            try:
                response = requests.get(url)
                
                if response.status_code == 200:
                    data = response.json()
                    articles = data['response']['docs']
                    
                    # Alleen relevante velden selecteren om CSV compact te houden
                    # Definieer je keywords
                    keyword_patterns = [
                                re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
                                for word in keywords
                            ]
                    for art in articles:
                        # Check of keywords voorkomen in de kop of de samenvatting
                        headline = art.get('headline', {}).get('main', "").lower()
                        abstract = art.get('abstract', "").lower()
                        
                        # Filter: check of één van de keywords in de tekst staat
                        if any(pattern.search(headline) for pattern in keyword_patterns) and art.get('section_name') in sections:
                            year_data.append({
                                'pub_date': art.get('pub_date'),
                                'headline': headline,
                                'abstract': abstract,
                                'section': art.get('section_name'),
                                'web_url': art.get('web_url')
                            })                
                elif response.status_code == 429:
                    print("  -> Rate limit bereikt! 60 seconden pauze...")
                    time.sleep(60)
                    # Je zou hier een 'retry' kunnen inbouwen
                else:
                    print(f"  -> Fout {response.status_code} bij {year}-{month}")

            except Exception as e:
                print(f"  -> Er ging iets mis: {e}")

            # Cruciaal: wacht 10 seconden tussen elke maand om 429 errors te voorkomen
            time.sleep(10)

        # Sla data per jaar op als een CSV
        if year_data:
            df = pd.DataFrame(year_data)
            # Verwijder duplicates op basis van headline
            df = df.drop_duplicates(subset=['headline'])
            filename = f"nyt_data_{year}.csv"
            df.to_csv(filename, index=False, encoding='utf-8')
            print(f"Jaar {year} succesvol opgeslagen in {filename} met {len(df)} articles✔️")

if __name__ == "__main__":
    if not API_KEY:
        print("Fout: Geen API_KEY gevonden in .env bestand.")
    else:
        fetch_and_save_nyt_data()

Ophalen: 2006-1...
Ophalen: 2006-2...
Ophalen: 2006-3...
Ophalen: 2006-4...
Ophalen: 2006-5...
Ophalen: 2006-6...
Ophalen: 2006-7...
Ophalen: 2006-8...
Ophalen: 2006-9...
Ophalen: 2006-10...
Ophalen: 2006-11...
Ophalen: 2006-12...
Jaar 2006 succesvol opgeslagen in nyt_data_2006.csv met 5079 articles✔️


In [67]:
df = pd.read_csv("./nyt_data_2006.csv", sep=",")


In [68]:
df['section'].unique()

<StringArray>
[       'World',     'New York',         'U.S.', 'Business Day',
  'Real Estate',   'Technology',   'Your Money',       'Health',
      'Science',    'Education',   'Washington',   'Job Market',
        'Blogs']
Length: 13, dtype: str